# ME 144/244: Modeling, Simulation, and Digital Twins of Drone-Based Systems
## Project 7: Aerial Firefighting
## Spring 2026 Semester


In [ ]:
# Reload edited .py files (e.g. simulation.py) automatically, without restarting the kernel
%load_ext autoreload
%autoreload 2

import pickle
import numpy as np
import matplotlib.pyplot as plt

from animation import Animation
from geneticalgorithm import GeneticAlgorithm
from aerial_sensitivity_analysis import AerialSensitivityAnalyzer

with open('parameters.pkl', 'rb') as f:
    parameters = pickle.load(f)


## Problem 2

In [ ]:
LAM_fixed = [ 6.93e1,  0.0 , 1.751e2 ,-1.192e1, 4.29e-3,  1.83e-1 , 2.159e-2 , 6.921e-1, 8.21,  4.878 , 4.85 , 1.5207e3]
try:
    from simulation import Simulation 
    sim_student = Simulation(parameters, LAM_fixed)
    hist_plane_s, hist_parts_s = sim_student.simulate_path_with_nozzles()
    cost_student = sim_student.calculate_cost()
    print(f'Student simulation cost (expect 0.0508): {cost_student:.4f}')
    if abs(cost_student - 0.0508) < 0.001:
        print('PASS — implementation matches reference.')
    else:
        print(f'MISMATCH — check your implementation (reference = 0.0508, yours = {cost_student:.4f}).')
except SyntaxError:
    print('student_simulation.py still has unfilled blanks (SyntaxError).')
    print('Complete all steps before running this cell.')
except Exception as e:
    print(f'Error running student simulation: {e}')

Results are saved to `ga_results.pkl` so you only need to run it once 

In [ ]:
# Run the GA (skip if ga_results.pkl already exists and you don't want to re-run)
GA = GeneticAlgorithm(parameters)
ga_data = GA.GA(print_bool=True)

with open('ga_results.pkl', 'wb') as f:
    pickle.dump(ga_data, f)

print('GA complete. Results saved to ga_results.pkl.')
print(f'Best cost:  {ga_data["best_cost"][-1]:.4f}')
print(f'Best LAM:   {ga_data["best_p_strings"][0]}')

In [ ]:
# Load saved GA results (use this instead of re-running the GA)
with open('ga_results.pkl', 'rb') as f:
    ga_data = pickle.load(f)

print(f'Best cost:  {ga_data["best_cost"][-1]:.4f}')
print(f'Best LAM:   {ga_data["best_p_strings"][0]}')

In [ ]:
# Convergence plot: best cost, average cost, parents average cost vs. generation
gens = range(1, len(ga_data['best_cost']) + 1)

plt.figure(figsize=(9, 4))
plt.plot(gens, ga_data['best_cost'],           label='Best cost')
plt.plot(gens, ga_data['average_cost'],        label='Average cost')
plt.plot(gens, ga_data['parents_average_cost'],label='Parents avg cost')
plt.xlabel('Generation')
plt.ylabel('Cost $\\Pi$')
plt.title('Problem 2(c): GA Convergence')
plt.legend()
plt.tight_layout()
plt.savefig('../animations/ga_convergence.png', dpi=150)
plt.show()
print('Convergence plot saved to ../animations/ga_convergence.png')

In [ ]:
# Run the best design and animate (uncomment animate line to save)
LAM_best = ga_data['best_p_strings'][0]
sim_best = Simulation(parameters, LAM_best)
hist_plane_best, hist_parts_best = sim_best.simulate_path_with_nozzles()

print(f'Best LAM:  {LAM_best}')
print(f'Best cost: {sim_best.calculate_cost():.4f}')

# Uncomment to generate the animation for the optimized design
anim_best = Animation(hist_plane_best, hist_parts_best, parameters)
anim_best.animate_firefighting(title='ga_result', savefig=True)

## Problem 3 

In [ ]:
TANK_VOLUME = parameters['TANK_VOLUME']  # 10 m^3

print(f"{'Radius (m)':<15} {'Vol (m^3)':<18} {'N_total':>12}")
print('-' * 47)
for r in [0.05, 0.1, 0.2, 0.01, 0.003, 0.001, 0.0003]:
    vol = (4/3) * np.pi * r**3
    N   = int(TANK_VOLUME / vol)
    print(f'{r:<15.4f} {vol:<18.6e} {N:>12,}')

## Problem 4 

In [ ]:
# Sensitivity sweep using the default baseline LAM from AerialSensitivityAnalyzer.default_base_lam()
analyzer = AerialSensitivityAnalyzer(parameters, animations_dir='../animations')
base_lam_default = AerialSensitivityAnalyzer.default_base_lam()

print('Running sensitivity sweep with default baseline LAM...')
print(f'Baseline LAM: {base_lam_default}')

results_default = analyzer.run_velocity_drop_velocity(
    base_lam=base_lam_default,
    make_plots=True,
    tag='default',
)


In [ ]:
# Sensitivity sweep using the GA best LAM
base_lam_ga = AerialSensitivityAnalyzer.load_ga_best_lam(
    ga_results_path='ga_results.pkl',
    fallback=AerialSensitivityAnalyzer.default_base_lam(),
)

print('Running sensitivity sweep with GA best LAM...')
print(f'GA best LAM: {base_lam_ga}')

results_ga = analyzer.run_velocity_drop_velocity(
    base_lam=base_lam_ga,
    make_plots=True,
    tag='ga',
)
print('Done. Saved sensitivity_surface_ga.png and sensitivity_heatmap_ga.png to ../animations/')

In [ ]:
LAM = [ 6.93e1,  0.0 , 1.751e2 ,-1.192e1, 4.29e-3,  1.83e-1 , 2.159e-2 , 6.921e-1, 8.21,  4.878 , 4.85 , 1.5207e3]

## Optional: save the optimized design as an MP4
The GIF saved in Problem 2(c) (../animations/ga_result.gif) is all you need for the report. This cell also tries to save an MP4, which needs ffmpeg. If you don't have ffmpeg, 

pip install imageio-ffmpeg

provides one without a system install. If this cell fails, you can skip it.

In [ ]:
import os
import matplotlib as mpl
from matplotlib import animation

# Prefer the ffmpeg bundled with imageio-ffmpeg if it's installed; otherwise use a system ffmpeg.
try:
    import imageio_ffmpeg
    mpl.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
except ImportError:
    pass

mp4_path = '../animations/ga_result.mp4'
if not animation.writers.is_available('ffmpeg'):
    print('ffmpeg not found, skipping MP4. Run `pip install imageio-ffmpeg` to enable it.')
else:
    try:
        anim_best.animate_firefighting(title='ga_result', savefig=True, video_format='mp4')
        print(f'Saved {mp4_path}')
    except Exception as e:
        if os.path.exists(mp4_path):
            os.remove(mp4_path)  # don't leave a half-written file behind
        print(f'Could not save MP4 ({type(e).__name__}). The GIF from Problem 2(c) is enough for the report.')
